# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Print main metadata fields
metadata_obj = dataset.metadata
print("Name:", getattr(metadata_obj, 'name', 'N/A'))
print("Description:", getattr(metadata_obj, 'description', 'N/A'))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List record sets, fields, and columns by their @id
print("Record sets in this dataset:")
for rs in dataset.metadata.record_sets:
    print(f"- record_set @id: {rs.id}, name: {getattr(rs, 'name', 'N/A')}")
    print("  Fields:")
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"    - field @id: {field.id}, name: {getattr(field, 'name', '')}, dataType: {getattr(field, 'data_type', '')}")
            # List columns for each field
            if hasattr(field, 'columns'):
                for col in field.columns:
                    print(f"       - column @id: {col.id}, name: {getattr(col, 'name', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Select record set @id(s) from the output of the previous cell
# For this dataset, let's assume primary data is in a record set with the @id 'https://api.app.sen.science/frontiers/7154140/4a6a0c2c-e6ad-4100-ab9a-26e82b1f345d'
record_sets = [
    'https://api.app.sen.science/frontiers/7154140/4a6a0c2c-e6ad-4100-ab9a-26e82b1f345d',  # Main protein table example
    # Add more record set @id's if available from the previous cell's output
]
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records from record_set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records. Columns: {dataframes[record_set_id].columns.tolist()}")

main_record_set_id = record_sets[0]
print("Columns in the main record set DataFrame:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Find a numeric field by inspecting columns above. Assume 'cr:peptide_count' exists and is numeric (update the field @id if needed).
# If unsure, print the columns in the previous cell to select an actual numeric field.
numeric_field = 'cr:peptide_count'  # Example numeric field @id
record_set_id = main_record_set_id

df = dataframes[record_set_id]
if numeric_field in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} out of {len(df)}")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field if available (e.g., 'cr:accession' or similar)
    group_field = 'cr:accession'  # Example group field @id
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
else:
    print(f"Field {numeric_field} not found in the DataFrame. Please update the numeric_field with a valid column @id.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Histogram of numeric field
if numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Scatter plot of two numeric fields if available
    other_numeric_field = 'cr:coverage_pct'  # Try another column @id. Update as necessary.
    if other_numeric_field in df.columns:
        plt.figure(figsize=(8,5))
        sns.scatterplot(x=df[numeric_field], y=df[other_numeric_field])
        plt.xlabel(numeric_field)
        plt.ylabel(other_numeric_field)
        plt.title(f'{numeric_field} vs {other_numeric_field}')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* The FAIR² Croissant dataset from mass spectrometry provides rich tabular data on protein characteristics, including accession numbers and peptide counts.
* We demonstrated how to load Croissant datasets using `mlcroissant`, inspect record sets, and dynamically extract data using `@id` references.
* Exploratory analysis included basic filtering, normalization, grouping, and plotting for numeric fields.
* For in-depth biological or biochemical analysis, refer to the metadata and field descriptions for specific proteins and modifications.